# SAC Irrigation Training - v2.11 (LayerNorm critic, Colab)

**Algorithm:** SAC (stable_baselines3) with VDN-factorised twin-Q critic + LayerNorm hidden layers
**Replay buffer:** SB3 standard 1-step `ReplayBuffer` (NOT the E3 NStepReplayBuffer)
**Single change vs v2.7:** LayerNorm inserted after each hidden Linear in the critic
**Gamma:** 0.99 (v2.7 baseline, UNCHANGED)

## Why v2.11 exists

Phase 1 (v2.10 E2/E3/E4) ruled out three approaches to the v2.7 deadly-triad cascade:
- E2 (TQC + k=5): quantile truncation is structurally inert under VDN-sum.
- E3 (TQC + n-step buffer): the buffer drops the soft-Bellman entropy bonus on intermediate steps, driving a negative Q-cascade earlier than v2.7.
- E4 (SAC + gamma=0.98): stops the cascade but produces a 6-mm/day flat policy (-16% wet-year yield vs v2.7) because 1/(1-gamma)=50 is shorter than the 93-day season.

v2.11 tests a fundamentally different cascade hypothesis. Yue et al. NeurIPS 2023 (arXiv:2310.04411) show via Neural Tangent Kernel analysis that Q-divergence in deep RL can be driven by network optimization dynamics (the Self-Excite Eigenvalue Measure, SEEM) and that **LayerNorm in the critic's hidden layers reliably suppresses SEEM** with no detrimental bias on the learned policy. Nauman et al. RLC 2024 (arXiv:2403.05996) confirm this in online RL on dm_control.

The v2.7 critic_loss trajectory is roughly factor-of-10 growth per 10k steps from step 150k onward (clean exponential across 12 orders of magnitude) - the signature LayerNorm is designed to suppress.

## Acceptance criterion

Primary (cascade suppressed):
- `|q_inflation_pct| < 30%` throughout 250k steps (v2.7 hits +200% at step 200k)
- `critic_loss` never exceeds ~50 past step 100k
- `actor/std/spatial` stays in [0.20, 0.40] throughout

Secondary (policy quality preserved):
- 9-cell yields within +-3% of v2.7 best_model (seed 0)

## Early-kill rule

If at any checkpoint past step 150k BOTH `q_inflation_pct > 100%` AND `actor/std/spatial < 0.15`, stop the run - LayerNorm did not suppress the cascade.


In [ ]:
# Cell 1: Mount Google Drive, clone repo, install deps.
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive mounted. Results -> {DRIVE_ROOT}')

if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', '/content/thesis'],
    check=True
)

os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')

# Install SB3 - sb3-contrib NOT required for v2.11 (no TQC).
subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0',
     'gymnasium', 'wandb', 'pytest'],
    check=True
)

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2: WandB secret + GPU check.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('     Training continues without WandB - add the key to Colab Secrets to enable it.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')


In [ ]:
# Cell 3: Pre-training validation.
#
# Runs smoke tests, factorized-critic tests (which include v2.11 LayerNorm
# critic shape and param-count guards), and a 1000-step pilot to catch
# import/wiring bugs.  Abort if anything fails.
#
# NOTE: TQC critic tests are intentionally NOT run here - v2.11 uses SAC.

import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 + v2.11 architectures)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'FACTORIZED CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check, ~1 minute)...')
from src.rl.train_v211 import train_sac_v211
_ = train_sac_v211(
    seed=999,
    output_dir='/content/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')


In [ ]:
# Cell 4: Full 250k training (SAC v2.11, LayerNorm critic, gamma=0.99).
# ~30-55 min on A100, ~2-2.5 h on T4.
#
# Start with SEED=0 (paired with v2.7 seed 0 published numbers).
# Expand to seeds 1, 2 only after seed-0 results meet the acceptance criterion.

SEED = 0       # CHANGE per session

from src.rl.train_v211 import train_sac_v211

model = train_sac_v211(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    gamma=0.99,      # v2.7 baseline; LayerNorm is the experimental variable
)
print('Training complete.')


In [ ]:
# Cell 5: Copy results to Google Drive (excluding replay buffer).
import shutil, os, datetime

src = f'/content/thesis/results/rl/sac_v211_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'{DRIVE_ROOT}/sac_v211_seed{SEED}_{timestamp}'

shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print(f'Saved to: {dst}')
print()
for root, _, files in os.walk(dst):
    for f in files:
        p = os.path.join(root, f)
        size = os.path.getsize(p)
        rel  = os.path.relpath(p, dst)
        print(f'  {rel}  ({size/1024:.1f} KB)')


In [ ]:
# Cell 6: Post-training 9-cell evaluation (SAC eval path).
#
# v2.11 produces a SAC checkpoint, so use exp_rl.py (not exp_rl_tqc.py).
# The runner auto-detects the LayerNorm critic via the 1-D 'critic.qf0.1.weight'
# key and dispatches to V211CTDESACPolicy.  The observation builder uses the
# v2.7 path (8 features/agent, 1097-dim) since v2.11's actor and obs layout
# are unchanged from v2.7.
import subprocess, sys

model_path = f'/content/thesis/results/rl/sac_v211_seed{SEED}/best_model/best_model.zip'

print('Evaluating on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',     'eval',
    '--model',    model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'

print('\nEvaluating on 9-cell grid (noisy forecast, seed=42)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',       'eval',
    '--model',      model_path,
    '--scenario',   'all',
    '--budget',     'all',
    '--forecast',   'noisy',
    '--noise-seed', '42',
], capture_output=False)
if r.returncode != 0:
    print('Noisy-forecast eval failed; perfect-forecast only.')


In [ ]:
# Cell 7: Q-inflation trajectory plot (v2.11 cascade diagnostic).
#
# With gamma=0.99 (UNCHANGED from v2.7), the structural baseline is the same:
#     Q_structural = alpha * geom_weight * (-log pi_start)
#     geom_weight = min(1/(1-gamma), 93) = min(100, 93) = 93
#     With alpha=0.05 and -log pi_start ~ 89:
#         Q_structural ~ 0.05 * 93 * 89 ~ 414
# v2.7 measured ~378-414 BEFORE the cascade (q_inflation_pct ~ -9% to 0%),
# then exploded to >+200% post step 175k.  v2.11 should stay <+30% throughout
# if LayerNorm has done its job.
#
# Healthy:         |q_inflation_pct| < 30%
# Cascade onset:   |q_inflation_pct| > 50% and trending
# Full cascade:    |q_inflation_pct| > 200% or sign flip with critic_loss spike

import pandas as pd
import matplotlib.pyplot as plt

csv_path = f'/content/thesis/results/rl/sac_v211_seed{SEED}/bias_ratio_log.csv'
df = pd.read_csv(csv_path)
print(df.tail(10))

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax = axes[0]
ax.plot(df['step'], df['q_pred_mean'],  '-o', label='Q_pred_mean',                  color='C0')
ax.plot(df['step'], df['q_structural'], '-s', label='Q_structural (theoretical)',   color='C1')
ax.axhline(0, color='k', linestyle=':', alpha=0.3)
ax.set_ylabel('Q value')
ax.set_title(f'v2.11 (SAC, LayerNorm critic, gamma=0.99) seed {SEED} - Q calibration')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(df['step'], df['q_inflation_pct'], '-o', color='C3', label='q_inflation_pct')
ax.axhline(0.0,   color='k',      linestyle='-',  alpha=0.4, label='ideal')
ax.axhline(30.0,  color='g',      linestyle=':',  alpha=0.6, label='acceptance threshold (30%)')
ax.axhline(-30.0, color='g',      linestyle=':',  alpha=0.6)
ax.axhline(50.0,  color='orange', linestyle=':',  alpha=0.6, label='cascade onset (50%)')
ax.axhline(-50.0, color='orange', linestyle=':',  alpha=0.6)
ax.axhline(200.0, color='r',      linestyle=':',  alpha=0.6, label='full cascade (200%)')
ax.axhline(-200.0,color='r',      linestyle=':',  alpha=0.6)
ax.set_ylabel('Q_inflation %')
ax.set_xlabel('training step')
ax.legend(loc='best', fontsize='small')
ax.grid(alpha=0.3)

plt.tight_layout()
plot_path = f'/content/thesis/results/rl/sac_v211_seed{SEED}/q_inflation_trajectory.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved plot: {plot_path}')


In [ ]:
# Cell 8: Resume from Drive checkpoint (if session was interrupted).
# Uncomment and fill in CHECKPOINT_STEP / CHECKPOINT_DRIVE_PATH.

# SEED  = 0
# CHECKPOINT_STEP = 100_000
# CHECKPOINT_DRIVE_PATH = f'{DRIVE_ROOT}/sac_v211_seed{SEED}_YYYYMMDD_HHMMSS'
#
# import shutil, os
# local_dir = f'/content/thesis/results/rl/sac_v211_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(CHECKPOINT_DRIVE_PATH, local_dir, dirs_exist_ok=True)
#
# from stable_baselines3 import SAC
# from src.rl.networks import V211CTDESACPolicy
# ckpt = f'{local_dir}/checkpoints/sac_v211_seed{SEED}_{CHECKPOINT_STEP}_steps.zip'
# model = SAC.load(ckpt, custom_objects={'policy_class': V211CTDESACPolicy})
# # Continue training with model.learn(total_timesteps=..., reset_num_timesteps=False)
